# Crash Course — fMRI 데이터 분석

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [crash-course/fmri.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/crash-course/fmri.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 최소 실행 예시

### 예시 1 — 2,000개 voxel에 같은 GLM을 돌리고 다중비교 보정하기

In [ ]:
import numpy as np
from scipy import stats
from scipy.stats import gamma
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(100)
TR, n, dt = 2.0, 160, 0.1
t_hrf = np.arange(0, 32, dt)
hrf = gamma.pdf(t_hrf, 6) - gamma.pdf(t_hrf, 16) / 6       # 이중 감마 모양 HRF
t_hi = np.arange(0, n * TR, dt)
box = ((t_hi % 40) < 20).astype(float)                       # 20초 과제 / 20초 쉼
task = (np.convolve(box, hrf)[: t_hi.size] * dt)[:: int(TR / dt)]
task /= task.max()

# 설계 행렬: 절편 + 과제 + 느린 drift를 잡는 코사인 2개
k = np.arange(n)
X = np.column_stack([np.ones(n), task] + [np.cos(np.pi * j * (k + 0.5) / n) for j in (1, 2)])

# 가상 voxel 2,000개 중 앞 100개만 과제에 반응 (% 신호 변화로 약 1%)
n_vox = 2000
amp = np.zeros(n_vox); amp[:100] = 1.0
Y = 100 + np.outer(task, amp) + rng.normal(0, 1.2, (n, n_vox))

# voxel마다 같은 GLM을 한꺼번에 푼다 (mass-univariate)
beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
resid = Y - X @ beta
dof = n - X.shape[1]
sigma2 = (resid ** 2).sum(0) / dof
c = np.array([0, 1, 0, 0])
t = (c @ beta) / np.sqrt(sigma2 * (c @ np.linalg.inv(X.T @ X) @ c))
p = stats.t.sf(t, dof)

for name, rej in [("보정 없음 p<0.05", p < 0.05),
                  ("Bonferroni", multipletests(p, 0.05, "bonferroni")[0]),
                  ("BH FDR", multipletests(p, 0.05, "fdr_bh")[0])]:
    print(f"{name:16} 검출 {rej.sum():4d} | 진짜 {rej[:100].sum():3d} | 가짜 {rej[100:].sum():3d}")

### 예시 2 — decoding에서 feature 선택 누수

In [ ]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(1)
y = np.tile([0, 1], 24)                      # 시행 48개, 두 조건
runs = np.repeat(np.arange(4), 12)           # run 4개
cv, clf = GroupKFold(4), LogisticRegression(max_iter=2000)

bad, good = [], []
for _ in range(20):                          # 신호 없는 자료를 20번 새로 만들어 평균을 본다
    X = rng.normal(size=(48, 2000))          # voxel 2,000개, 조건과 무관
    top = SelectKBest(f_classif, k=20).fit(X, y).get_support()             # ✗ 전체 자료로 voxel 고르기
    bad.append(cross_val_score(clf, X[:, top], y, cv=cv, groups=runs).mean())
    good.append(cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), clf),  # ✓ fold 안에서 고르기
                                X, y, cv=cv, groups=runs).mean())
print("우연 수준 0.50 | 누수 있는 정확도 평균 %.2f | 누수 없는 정확도 평균 %.2f (범위 %.2f–%.2f)"
      % (np.mean(bad), np.mean(good), min(good), max(good)))